In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 9
fig_height = 6
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L2hvbWUvbWl0aHVubWFuaXZhbm5hbi9wcm9qZWN0cy9iZW5jaG1hcmtpbmdfbG9zc19mdW5jdGlvbnNfZWNnX3JlY29uc3RydWN0aW9uL2Jvb2s='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/usr/lib/python3.12/importlib/_bootstrap.py": 1781873160.0, "/usr/lib/python3.12/importlib/_bootstrap_external.py": 1781873160.0, "/usr/lib/python3.12/zipimport.py": 1781873160.0, "/usr/lib/python3.12/codecs.py": 1781873160.0, "/usr/lib/python3.12/encodings/aliases.py": 1781873160.0, "/usr/lib/python3.12/encodings/__init__.py": 1781873160.0, "/usr/lib/python3.12/encodings/utf_8.py": 1781873160.0, "/usr/lib/python3.12/abc.py": 1781873160.0, "/usr/lib/python3.12/io.py": 1781873160.0, "/usr/lib/python3.12/stat.py": 1781873160.0, "/usr/lib/python3.12/_collections_abc.py": 1781873160.0, "/usr/lib/python3.12/genericpath.py": 1781873160.0, "/usr/lib/python3.12/posixpath.py": 1781873160.0, "/usr/lib/python3.12/os.py": 1781873160.0, "/usr/lib/python3.12/_sitebuiltins.py": 1781873160.0, "/usr/lib/python3.12/__future__.py": 1781873160.0, "/usr/lib/python3.12/warnings.py": 1781873160.0, "/usr/lib/python3.12/importlib/__init__.py": 1781873160.0, "/usr/lib/python3.12/importlib/machinery.py": 17818

In [2]:
#| label: didactic-sqi-noise-simulation
#| fig-cap: DIDACTIC SIMULATION — stipulated ECG-like signal and synthetic corruptions; not a model robustness result.
import numpy as np
import scipy.signal as signal
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def compute_sqi_metrics(ecg_signal, fs=500):
    """
    Calculates kSQI, sSQI, and pSQI for a 1D ECG signal.
    """
    # 1. Kurtosis SQI
    ksqi = stats.kurtosis(ecg_signal, fisher=False)
    
    # 2. Skewness SQI
    ssqi = stats.skew(ecg_signal)
    
    # 3. Power Spectrum SQI (pSQI)
    freqs, psd = signal.welch(ecg_signal, fs=fs, nperseg=fs)
    
    qrs_power = np.sum(psd[(freqs >= 5) & (freqs <= 15)])
    total_power = np.sum(psd[(freqs >= 5) & (freqs <= 40)]) + 1e-8
    psqi = qrs_power / total_power
    
    return {"kSQI": round(ksqi, 3), "sSQI": round(ssqi, 3), "pSQI": round(psqi, 3)}

def inject_noise_stress(ecg_signal, snr_db=12, noise_type="gaussian", fs=500, rng=None):
    """
    Injects synthetic AWGN or Baseline Wander at a target SNR in dB.
    """
    p_signal = np.mean(ecg_signal ** 2)
    t = np.linspace(0, len(ecg_signal) / fs, len(ecg_signal))
    
    rng = np.random.default_rng(20260801) if rng is None else rng
    if noise_type == "gaussian":
        p_noise = p_signal / (10 ** (snr_db / 10.0))
        noise = rng.normal(0, np.sqrt(p_noise), len(ecg_signal))
    elif noise_type == "baseline_wander":
        # Respiratory drift at 0.3 Hz
        drift = np.sin(2 * np.pi * 0.3 * t)
        p_drift = np.mean(drift ** 2)
        scale = np.sqrt((p_signal / (10 ** (snr_db / 10.0))) / p_drift)
        noise = scale * drift
    else:
        raise ValueError(f"Unknown noise type: {noise_type}")
        
    return ecg_signal + noise

# Simulate a deterministic ECG-like teaching signal. This is not an acquired ECG.
fs = 500
t = np.linspace(0, 2.0, 1000)
clean_ecg = np.sin(2 * np.pi * 1.2 * t) ** 21 + 0.2 * np.sin(2 * np.pi * 1.2 * t + 0.8) ** 3

# Inject AWGN (12 dB) and Baseline Wander (6 dB)
rng = np.random.default_rng(20260801)
noisy_awgn = inject_noise_stress(clean_ecg, snr_db=12, noise_type="gaussian", fs=fs, rng=rng)
noisy_bw = inject_noise_stress(clean_ecg, snr_db=6, noise_type="baseline_wander", fs=fs, rng=rng)

# Compute SQI Metrics
sqi_clean = compute_sqi_metrics(clean_ecg, fs=fs)
sqi_awgn = compute_sqi_metrics(noisy_awgn, fs=fs)
sqi_bw = compute_sqi_metrics(noisy_bw, fs=fs)

print("==================================================")
print(f"Clean ECG SQI Metrics:     {sqi_clean}")
print(f"AWGN 12dB Corrupted SQI:    {sqi_awgn}")
print(f"Baseline Drift 6dB SQI:    {sqi_bw}")
print("==================================================")

Clean ECG SQI Metrics:     {'kSQI': np.float64(4.554), 'sSQI': np.float64(0.023), 'pSQI': np.float64(1.0)}
AWGN 12dB Corrupted SQI:    {'kSQI': np.float64(4.383), 'sSQI': np.float64(0.042), 'pSQI': np.float64(0.966)}
Baseline Drift 6dB SQI:    {'kSQI': np.float64(4.611), 'sSQI': np.float64(0.119), 'pSQI': np.float64(1.0)}


In [3]:
fig_noise = make_subplots(
    rows=3, cols=1, 
    subplot_titles=(
        f"Clean Signal (kSQI: {sqi_clean['kSQI']}, pSQI: {sqi_clean['pSQI']})", 
        f"AWGN 12dB Corrupted (kSQI: {sqi_awgn['kSQI']}, pSQI: {sqi_awgn['pSQI']})", 
        f"Baseline Drift 6dB (kSQI: {sqi_bw['kSQI']}, pSQI: {sqi_bw['pSQI']})"
    )
)

fig_noise.add_trace(go.Scatter(x=t, y=clean_ecg, mode='lines', line=dict(color='#10b981')), row=1, col=1)
fig_noise.add_trace(go.Scatter(x=t, y=noisy_awgn, mode='lines', line=dict(color='#f43f5e')), row=2, col=1)
fig_noise.add_trace(go.Scatter(x=t, y=noisy_bw, mode='lines', line=dict(color='#fbbf24')), row=3, col=1)

fig_noise.update_layout(
    title="DIDACTIC SIMULATION — Noise and SQI Response",
    template="plotly_dark",
    height=600, margin=dict(l=20, r=20, t=60, b=20),
    showlegend=False
)
fig_noise.show()